# Dragapult policy prior — kendi replay datasetimizle

Bu notebook **bizim ham `episode-*.json` replay datasetimizi** kullanır; Mazlum’un iyi taraflarını alır:
- option feature’ları + card ID embeddings
- masked cross-entropy
- legal option masking
- küçük policy modeli

Eğitim verisi: sadece **bizim Dragapult maçlarımız**. Mazlum’un hazır Dragapult dataseti kullanılmaz.

## Gerekli Kaggle girdileri
1. `ptcg-replays` — ham replay JSON’ları (veya zip).
2. `ptcg-code` — repo code dataset’i: içinde `src/mazlum/build_dataset.py`, `src/mazlum/train_bc.py`, `src/mazlum/features.py`, `src/mazlum/model.py` ve ideal olarak `vendor/cg`.

`vendor/cg` yoksa attack damage/energy feature’ları boş kalır; eğitim yine çalışır ama daha zayıf olur.

## Çıktı
- `/kaggle/working/dataset_dragapult_ours.npz`
- `/kaggle/working/policy_prior.npz`


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
WORK.mkdir(parents=True, exist_ok=True)

# Find/extract code dataset. A single zip upload is supported directly.
code_zips = sorted(INPUT.rglob('ptcg-code.zip'))
extracted_code = WORK / 'ptcg-code'
if code_zips and not (extracted_code / 'src/mazlum/build_dataset.py').exists():
    print('extracting', code_zips[0])
    with zipfile.ZipFile(code_zips[0]) as zf:
        zf.extractall(extracted_code)

builders = []
for root in (extracted_code, INPUT):
    if root.exists():
        builders.extend(root.rglob('src/mazlum/build_dataset.py'))
        builders.extend(root.rglob('build_dataset.py'))
if not builders:
    raise RuntimeError('build_dataset.py bulunamadı. ptcg-code datasetini/zipini ekle.')
BUILD = builders[0]
CODE_ROOT = BUILD.parents[2] if BUILD.parent.name == 'mazlum' else BUILD.parent
TRAIN = None
for root in (extracted_code, INPUT):
    if root.exists():
        TRAIN = next(iter(root.rglob('src/mazlum/train_bc.py')), None) or next(iter(root.rglob('train_bc.py')), None)
        if TRAIN is not None:
            break
if TRAIN is None:
    raise RuntimeError('train_bc.py bulunamadı.')
print('BUILD', BUILD)
print('TRAIN', TRAIN)
print('CODE_ROOT', CODE_ROOT)

# Find raw replays, extracting zip JSONs if needed
episodes = list(INPUT.rglob('episode-*.json'))
if episodes:
    REPLAYS = INPUT
else:
    REPLAYS = WORK / 'raw_replays'
    REPLAYS.mkdir(parents=True, exist_ok=True)
    for z in sorted(INPUT.rglob('*.zip')):
        print('extracting', z)
        with zipfile.ZipFile(z) as zf:
            for m in zf.namelist():
                if not m.lower().endswith('.json') or m.endswith('/'): continue
                dest = REPLAYS / Path(m).name
                if dest.exists(): continue
                with zf.open(m) as src, open(dest, 'wb') as dst:
                    shutil.copyfileobj(src, dst, 1024*1024)
    episodes = list(REPLAYS.rglob('episode-*.json'))
print('raw episodes', len(episodes))

## 1. Sadece bizim Dragapult maçlarını seç

In [ ]:
from collections import Counter

OUR = 'Attention Is All You Mew'
def extract_decks(e):
    try:
        a = e['steps'][0][0]['visualize'][0]['action']
        return [list(map(int, d)) for d in a] if isinstance(a, list) and len(a) == 2 else None
    except (KeyError, IndexError, TypeError):
        return None
def is_dp(deck):
    return Counter(deck).get(121, 0) >= 2

selected = WORK / 'our_dragapult_replays'
selected.mkdir(parents=True, exist_ok=True)
allowed = {}
for p in REPLAYS.rglob('episode-*.json'):
    try:
        e = json.loads(p.read_text(encoding='utf-8'))
    except Exception:
        continue
    names = ((e.get('info') or {}).get('TeamNames') or [])[:2]
    ours = [i for i, n in enumerate(names) if OUR in str(n)]
    if len(ours) != 1:
        continue
    decks = extract_decks(e)
    if not decks:
        continue
    me = ours[0]
    if not is_dp(decks[me]):
        continue
    eid = (e.get('info') or {}).get('EpisodeId')
    if eid is None:
        continue
    eid = int(eid)
    dest = selected / f'episode-{eid}-replay.json'
    if not dest.exists():
        try:
            os.symlink(p, dest)
        except OSError:
            shutil.copy2(p, dest)
    allowed[eid] = [me]
print('selected episodes', len(allowed), 'files', len(list(selected.glob('*.json'))))

## 2. Mazlum feature/model hattıyla dataset üret

In [ ]:
raw_npz = WORK / 'dataset_dragapult_all.npz'
cmd = [sys.executable, str(BUILD), '--replays', str(selected), '--out', str(raw_npz), '--val-fraction', '0.2']
print(' '.join(cmd)); subprocess.run(cmd, check=True)

## 3. Sadece bizim oyuncu satırlarını tut
Not: `build_dataset.py` her iki oyuncunun kararlarını üretir; burada rakibin değil, bizim Dragapult aksiyonlarımızı tutuyoruz.

In [ ]:
import numpy as np
d = np.load(raw_npz, mmap_mode='r')
rows = {k: d[k] for k in d.files}
n = rows['player'].shape[0]
keep = np.array([int(rows['player'][i]) in allowed.get(int(rows['episode_id'][i]), []) for i in range(n)], dtype=bool)
print('rows', n, 'keep', int(keep.sum()))
out_npz = WORK / 'dataset_dragapult_ours.npz'
np.savez_compressed(out_npz, **{k: v[keep] for k, v in rows.items()})
print('wrote', out_npz, round(out_npz.stat().st_size/1e6, 2), 'MB')

## 4. Policy prior eğitimi

In [ ]:
prior_path = WORK / 'policy_prior.npz'
cmd = [sys.executable, str(TRAIN), '--dataset', str(out_npz), '--out', str(prior_path), '--epochs', '10', '--batch-size', '64', '--lr', '0.001', '--arena-every', '0']
print(' '.join(cmd)); subprocess.run(cmd, check=True)

In [ ]:
from IPython.display import FileLink, display
print('exists', prior_path.exists(), 'MB', round(prior_path.stat().st_size/1e6, 2) if prior_path.exists() else None)
if prior_path.exists():
    with np.load(prior_path) as z:
        print('arrays', len(z.files), 'params', sum(z[k].size for k in z.files))
    display(FileLink(str(prior_path)))

## Not
Bu model doğrudan submission değil. Sonraki adımda lean4 search’te **candidate prior** olarak kullanılacak; lethal override ve kural policy fallback korunacak.
